<a href="https://colab.research.google.com/github/stefanogiagu/corso_AML_2026/blob/main/utilis_GNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import glob

import torch

print(torch.__version__)


import torch.nn as nn
import torch.nn.functional as F


def load_data(datafiles,PLOT_ALL=False):
    # data is stored in hirearchical data format (h5), a file format desgined to store and maniuplate large size data structures
    # .h5 files can be accesses in python using the h5py library (https://docs.h5py.org/en/stable/)

    # read dataset (only 50k events to keep training time ~20' on google colab, better performance can be obtained adding the three commented files
    target = np.array([])
    p_data = np.array([])
    #datafiles =datafiles[0:10]


    # let's print what is contained in one of the files:
    f = h5py.File(datafiles[0])
    print(list(f.keys()))
    f.close()

    # each file contains different numpy arrays, the ones we are interested in are:
    # "jetConstituentsList": containing for each jet, and for each jet constituent particle (up to 150 particles) the 16 features associated to the jet particle
    # "jets": containing for each jet several (59) global features of the jet, we are here interested in the elements from -6:-1 that provide a onehot encoding of the jet-type label ['j_g', 'j_q', 'j_w', 'j_z', 'j_t]


    #loop over the files and concatenate the content to the p_adat and target arrays
    for fileIN in datafiles:
        print("Appending %s" %fileIN)
        f = h5py.File(fileIN) #read the h5 file
        data = np.array(f.get("jetConstituentList")) #jet constituents
        targ = np.array(f.get('jets')[0:,-6:-1])  #select ['j_g', 'j_q', 'j_w', 'j_z', 'j_t] out of the 59 features presents in the container
        p_data = np.concatenate([p_data, data], axis=0) if p_data.size else data
        target = np.concatenate([target, targ], axis=0) if target.size else targ
        del data, targ
        f.close()

    print(target.shape, p_data.shape)

    p_featurenames = ['j1_px','j1_py','j1_pz','j1_e','j1_erel','j1_pt','j1_ptrel',
     'j1_eta','j1_etarel','j1_etarot','j1_phi','j1_phirel','j1_phirot',
     'j1_deltaR','j1_costheta','j1_costhetarel']

    print(p_featurenames)
    labels = ['gluon', 'quark', 'W', 'Z', 'top']
    labels = ['gluon', 'quark', 'W', 'Z', 'top']

    print('target: ', target[0])
    print('so it\'s a jet of type: ',  labels[np.argmax(target[0])])
    print()

    print('jet - particle 0 - pT / eta / phi: ', p_data[0,0,5], ' / ', p_data[0,0,7], ' / ', p_data[0,0,10])
    if PLOT_ALL:
        plot_all_features(p_data, p_featurenames,labels,target)

    return p_featurenames, p_data, target,labels



def makePlot_p(feature_index, input_data, input_featurenames, labels,target):
    plt.subplots()
    for i in range(len(labels)):
        my_data = input_data[:,:,feature_index]
        # notice the use of numpy masking to select specific classes of jets
        my_data = my_data[np.argmax(target, axis=1) == i]
        # then plot the right quantity for the reduced array
        plt.hist(my_data[:,feature_index].flatten(), 50, density=True, histtype='step', fill=False, linewidth=1.5)
    plt.yscale('log', nonpositive='clip', )
    plt.legend(labels, fontsize=12, frameon=False)
    plt.xlabel(input_featurenames[feature_index], fontsize=15)
    plt.ylabel('Prob. Density (a.u.)', fontsize=15)
    plt.show()

def plot_all_features(p_data, p_featurenames,labels,target):
  # we now plot all the features
  for i in range(len(p_featurenames)):
      makePlot_p(i, p_data, p_featurenames, labels,target)


from sklearn.preprocessing import MinMaxScaler ,StandardScaler
def prepare_data(p_data, p_featurenames, target,N_part=100,NORMALIZE=True):
    # convert one-hot lables in integers labels, and split data

    p_label = np.argmax(target, axis=1)

    data_notnorm = np.array(p_data)
    print(data_notnorm.shape)
    #ADD sin phi, cos phi
    p_featurenames =p_featurenames+["j1_sin_phi","j1_cos_phi"]
    data_notnorm = np.concatenate([data_notnorm, np.expand_dims(np.sin(data_notnorm[:,:,10]),-1),np.expand_dims(np.cos(data_notnorm[:,:,10]),-1)],axis=2)

    # clear p_data
    p_data = []

    # training, validation, test split
    from sklearn.model_selection import train_test_split
    training_frac = 0.6 # training set fraction wrt whole sample
    testset_frac = 0.25 # test set fraction wrt validation set

    X_train,X_test,Y_train,Y_test = train_test_split(data_notnorm,p_label,train_size=training_frac, shuffle=True, random_state=1234)
    X_vali,X_test,Y_vali,Y_test = train_test_split(X_test,Y_test,test_size=testset_frac, shuffle=True, random_state=1234)

    print("original data shape")
    print(X_train.shape)
    print(X_vali.shape)
    print(X_test.shape)
    print(Y_train.shape)
    print(Y_vali.shape)
    print(Y_test.shape)

    print("reduced data shape")
    N_part = 100
    X_train = X_train[:,:N_part,:]# cut to N_part nodes
    X_vali = X_vali[:,:N_part,:]
    X_test = X_test[:,:N_part,:]
    print(X_train.shape)
    print(X_vali.shape)
    print(X_test.shape)

    scaler = StandardScaler()
    if NORMALIZE:
        #NOTE: StandardScaler works with (nevents, features) shaped arrays so let's convert (N, 50, 16) -> (N*50, 16) and back
        X_train_norm = scaler.fit_transform(X_train.reshape(-1,X_train.shape[-1])).reshape(X_train.shape)
        X_vali_norm = scaler.transform(X_vali.reshape(-1,X_vali.shape[-1])).reshape(X_vali.shape)
        X_test_norm = scaler.transform(X_test.reshape(-1,X_test.shape[-1])).reshape(X_test.shape)
        return X_train_norm,X_vali_norm,X_test_norm,Y_train,Y_vali,Y_test,p_featurenames, scaler

    return X_train,X_vali,X_test,Y_train,Y_vali,Y_test,p_featurenames, None



# -------------------------
# Dataset
# -------------------------
import copy
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, knn_graph, global_max_pool

class JetGraphDataset(Dataset):
    """
    Each sample is one jet: nodes are constituents
    x: node features (F = 18 expected)
    pos: [eta, sin(phi), cos(phi)] for kNN edges
    y: integer class label in [0..4]
    """
    def __init__(self, X: np.ndarray, y: np.ndarray,
                 k: int = 8,
                 idx_eta: int = 7,
                 idx_phi: int = 10,
                 idx_sinphi: int = 16,
                 idx_cosphi: int = 17):
        super().__init__()
        assert X.ndim == 3, f"Expected X shape (N, num_nodes, num_features), got {X.shape}"
        assert len(X) == len(y)
        self.X = X
        self.y = y.astype(np.int64)
        self.k = int(k)

        # indices in your feature tensor
        self.idx_eta = idx_eta
        self.idx_phi = idx_phi
        self.idx_sinphi = idx_sinphi
        self.idx_cosphi = idx_cosphi

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx: int) -> Data:
        x_np = self.X[idx]  # (num_nodes, num_features)
        y_np = self.y[idx]

        x = torch.tensor(x_np, dtype=torch.float)

        # pos = [eta, sin(phi), cos(phi)]
        # Use your standardized columns for eta and the standardized sin/cos you appended.
        # If you prefer non-standardized sin/cos for geometry, adapt accordingly.
        pos = torch.stack([
            x[:, self.idx_eta],
            x[:, self.idx_sinphi],
            x[:, self.idx_cosphi],
        ], dim=1).contiguous()  # (num_nodes, 3)

        # Build kNN edges (within this single graph)
        # loop=False avoids self-edges
        edge_index = knn_graph(pos, k=self.k, loop=False)

        data = Data(
            x=x,
            pos=pos,
            edge_index=edge_index,
            y=torch.tensor(y_np, dtype=torch.long)
        )
        return data

# -------------------------
# Model
# -------------------------
class GCNJetClassifier(nn.Module):
    def __init__(self, in_dim: int, emb_dim: int, num_layers: int, num_classes: int):
        super().__init__()
        assert num_layers >= 1

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.res_projs = nn.ModuleList()

        # layer 0: in_dim -> emb_dim
        self.convs.append(GCNConv(in_dim, emb_dim))
        self.bns.append(nn.BatchNorm1d(emb_dim))
        self.res_projs.append(nn.Linear(in_dim, emb_dim, bias=False))  # projection for residual

        # layers 1..: emb_dim -> emb_dim
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(emb_dim, emb_dim))
            self.bns.append(nn.BatchNorm1d(emb_dim))
            self.res_projs.append(nn.Identity())  # residual is already emb_dim

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim*2, 32), # funnel to mean + global graph desription
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(32, num_classes),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        for conv, bn, proj in zip(self.convs, self.bns, self.res_projs):
            x_in = x
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x, inplace=True)
            x = x + proj(x_in)  # residual add

        g = global_mean_pool(x, batch)
        m = global_max_pool(x, batch)
        gm = torch.concatenate([g,m],dim=-1)
        gm = F.dropout(gm, p=0.25, training=self.training)
        return self.mlp(gm)

# -------------------------
# Train / Eval utilities
# -------------------------
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    total = 0
    correct = 0
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    for batch in loader:
        batch = batch.to(device)
        logits = model(batch)
        loss = criterion(logits, batch.y)
        total_loss += float(loss.item()) * batch.num_graphs

        preds = logits.argmax(dim=1)
        correct += int((preds == batch.y).sum().item())
        total += int(batch.num_graphs)

    avg_loss = total_loss / max(total, 1)
    acc = correct / max(total, 1)
    return avg_loss, acc

def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer, device: torch.device):
    model.train()
    criterion = nn.CrossEntropyLoss()

    total = 0
    total_loss = 0.0
    correct = 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad(set_to_none=True)

        logits = model(batch)
        loss = criterion(logits, batch.y)
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item()) * batch.num_graphs
        preds = logits.argmax(dim=1)
        correct += int((preds == batch.y).sum().item())
        total += int(batch.num_graphs)

    avg_loss = total_loss / max(total, 1)
    acc = correct / max(total, 1)
    return avg_loss, acc
